# V2 — تحلیل خطای نهایی و inference مستقل A2-MP

تصمیم رسمی مدل از probability خام با aggregation برابر top3_mean و threshold برابر 0.40 گرفته می‌شود. probability calibrated برای گزارش عدم‌قطعیت نگه داشته می‌شود، ولی نباید برچسب تصمیم را تغییر دهد.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd

DATA_ROOT = Path(r'P:\\NexarCollisionData')
VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
MODEL_DIR = DATA_ROOT / 'models_v2'
INFERENCE_DIR = DATA_ROOT / 'inference_v2'
CALIBRATED_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_multipos_calibrated_validation_predictions.csv'
WINDOW_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_multipos_validation_sliding_window_predictions.csv'
CALIBRATION_REPORT_PATH = INFERENCE_DIR / 'a2_multipos_calibration_report.json'
ERROR_ANALYSIS_PATH = INFERENCE_DIR / 'a2_multipos_final_error_analysis.csv'
METADATA_BREAKDOWN_PATH = INFERENCE_DIR / 'a2_multipos_final_error_by_metadata.csv'
SUMMARY_PATH = INFERENCE_DIR / 'a2_multipos_final_error_summary.json'
ERROR_IMAGE_DIR = INFERENCE_DIR / 'error_analysis_a2_multipos_final'
ERROR_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MAX_EXAMPLES_PER_ERROR_TYPE = 12
FRAMES_PER_VISUALIZATION = 4

assert VIDEO_MANIFEST_PATH.exists(), 'Run notebook 07 first.'
assert CALIBRATED_PREDICTIONS_PATH.exists(), 'Run notebook 23 first.'
assert WINDOW_PREDICTIONS_PATH.exists(), 'Run notebook 21 first.'
assert CALIBRATION_REPORT_PATH.exists(), 'Run notebook 23 first.'

In [2]:
calibration_report = json.loads(CALIBRATION_REPORT_PATH.read_text(encoding='utf-8'))
selected_aggregation = calibration_report['aggregation']
raw_threshold = float(calibration_report['raw_selected_threshold_from_full_video_evaluation'])
temperature = float(calibration_report['final_temperature_fitted_on_all_validation_videos'])
calibrated_threshold = float(calibration_report['calibrated_threshold_equivalent_to_raw_decision'])

manifest = pd.read_csv(VIDEO_MANIFEST_PATH).copy()
predictions = pd.read_csv(CALIBRATED_PREDICTIONS_PATH).copy()
window_predictions = pd.read_csv(WINDOW_PREDICTIONS_PATH).copy()
for table in (manifest, predictions, window_predictions):
    table['video_id'] = table['video_id'].astype(str)
manifest['time_of_event'] = pd.to_numeric(manifest['time_of_event'], errors='coerce')
predictions['label'] = predictions['label'].astype(int)
predictions['raw_prediction'] = predictions['raw_prediction'].astype(int)
predictions['final_calibrated_prediction'] = predictions['final_calibrated_prediction'].astype(int)

metadata_columns = ['video_id', 'video_path', 'weather', 'light_conditions', 'scene', 'time_of_event', 'duration']
analysis = predictions.merge(manifest[metadata_columns], on='video_id', how='left', validate='one_to_one', suffixes=('', '_manifest'))
assert len(analysis) == 120
assert analysis['label'].isin([0, 1]).all()
assert (analysis['raw_prediction'] == (analysis['raw_video_probability'] >= raw_threshold).astype(int)).all()
assert (analysis['raw_prediction'] == analysis['final_calibrated_prediction']).all()

def error_type(row: pd.Series) -> str:
    if row.label == 1 and row.raw_prediction == 1:
        return 'true_positive'
    if row.label == 0 and row.raw_prediction == 0:
        return 'true_negative'
    if row.label == 0 and row.raw_prediction == 1:
        return 'false_positive'
    return 'false_negative'

analysis['error_type'] = analysis.apply(error_type, axis=1)
analysis['is_error'] = analysis['error_type'].isin(['false_positive', 'false_negative'])
analysis['max_window_contains_event'] = (
    analysis['time_of_event'].notna()
    & analysis['time_of_event'].between(analysis['max_window_start'], analysis['max_window_end'])
)
analysis['max_window_center_abs_error_seconds'] = np.where(
    analysis['time_of_event'].notna(),
    (analysis['time_of_event'] - analysis['max_window_center']).abs(),
    np.nan,
)
analysis = analysis.sort_values(['is_error', 'raw_video_probability'], ascending=[False, False]).reset_index(drop=True)
analysis.to_csv(ERROR_ANALYSIS_PATH, index=False)

print({'aggregation': selected_aggregation, 'raw_threshold': raw_threshold, 'temperature': temperature, 'calibrated_threshold': calibrated_threshold})
display(analysis['error_type'].value_counts().rename_axis('error_type').to_frame('videos'))
display(analysis.loc[analysis['is_error'], ['video_id', 'error_type', 'label', 'raw_prediction', 'raw_video_probability', 'final_calibrated_probability', 'max_window_start', 'max_window_end', 'time_of_event']].head(20))

{'aggregation': 'top3_mean', 'raw_threshold': 0.4, 'temperature': 1.4084302186965942, 'calibrated_threshold': 0.4285218762661993}


,videos
error_type,
true_positive,52
true_negative,31
false_positive,29
false_negative,8


,video_id,error_type,label,raw_prediction,raw_video_probability,final_calibrated_probability,max_window_start,max_window_end,time_of_event
0,1722,false_positive,0,1,0.937206,0.872048,13.0,18.0,NaN
1,1497,false_positive,0,1,0.928606,0.860747,17.5,22.5,NaN
2,1904,false_positive,0,1,0.900789,0.827255,27.5,32.5,NaN
3,1277,false_positive,0,1,0.809539,0.736411,5.0,10.0,NaN
4,1673,false_positive,0,1,0.783159,0.713364,20.0,25.0,NaN
5,1870,false_positive,0,1,0.772333,0.704184,0.0,5.0,NaN
6,1080,false_positive,0,1,0.760889,0.694635,7.5,12.5,NaN
7,1271,false_positive,0,1,0.758073,0.692308,10.0,15.0,NaN
8,2000,false_positive,0,1,0.751987,0.687310,7.5,12.5,NaN
9,1970,false_positive,0,1,0.697408,0.644019,35.0,40.0,NaN


In [3]:
metadata_rows = []
for column in ('weather', 'light_conditions', 'scene'):
    for value, group in analysis.groupby(column, dropna=False):
        metadata_rows.append({
            'metadata_field': column,
            'metadata_value': 'missing' if pd.isna(value) else value,
            'videos': int(len(group)),
            'false_positive': int(group['error_type'].eq('false_positive').sum()),
            'false_negative': int(group['error_type'].eq('false_negative').sum()),
            'errors': int(group['is_error'].sum()),
            'error_rate': float(group['is_error'].mean()),
        })
metadata_breakdown = pd.DataFrame(metadata_rows).sort_values(['metadata_field', 'error_rate', 'videos'], ascending=[True, False, False])
metadata_breakdown.to_csv(METADATA_BREAKDOWN_PATH, index=False)

positive_analysis = analysis.loc[analysis['label'].eq(1)].copy()
summary = {
    'model': 'A2-MP ResNet18 frozen + mean-max pooling',
    'official_decision_probability': 'raw top3_mean video probability',
    'selected_aggregation': selected_aggregation,
    'raw_threshold': raw_threshold,
    'temperature_for_probability_reporting': temperature,
    'calibrated_threshold_equivalent_to_raw_decision': calibrated_threshold,
    'true_positive': int(analysis['error_type'].eq('true_positive').sum()),
    'true_negative': int(analysis['error_type'].eq('true_negative').sum()),
    'false_positive': int(analysis['error_type'].eq('false_positive').sum()),
    'false_negative': int(analysis['error_type'].eq('false_negative').sum()),
    'uncertain_videos': int(analysis['uncertain'].sum()),
    'positive_max_window_contains_event_rate': float(positive_analysis['max_window_contains_event'].mean()),
    'positive_max_window_center_mae_seconds': float(positive_analysis['max_window_center_abs_error_seconds'].mean()),
    'note': 'Localization fields are supplementary diagnostics only; the official task is video-level accident detection.',
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Error analysis: {ERROR_ANALYSIS_PATH}')
print(f'Metadata breakdown: {METADATA_BREAKDOWN_PATH}')
print(f'Summary: {SUMMARY_PATH}')
display(metadata_breakdown.head(20))

Error analysis: P:\NexarCollisionData\inference_v2\a2_multipos_final_error_analysis.csv
Metadata breakdown: P:\NexarCollisionData\inference_v2\a2_multipos_final_error_by_metadata.csv
Summary: P:\NexarCollisionData\inference_v2\a2_multipos_final_error_summary.json


,metadata_field,metadata_value,videos,false_positive,false_negative,errors,error_rate
3,light_conditions,Dark,3,0,1,1,0.333333
4,light_conditions,Normal,108,27,7,34,0.314815
5,light_conditions,Twilight,9,2,0,2,0.222222
6,scene,Highway,37,8,6,14,0.378378
9,scene,Sub-urban,25,7,2,9,0.360000
8,scene,Other,3,1,0,1,0.333333
10,scene,Urban,53,13,0,13,0.245283
7,scene,Industrial,2,0,0,0,0.000000
1,weather,Cloudy,44,11,4,15,0.340909
2,weather,Rain,9,3,0,3,0.333333


In [4]:
def read_bgr_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float):
    step = 1.0 / fps if fps > 0 else 1.0 / 30.0
    for offset in (0.0, step, -step):
        cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, timestamp + offset) * 1000.0)
        ok, image = cap.read()
        if ok and image is not None:
            return image
    return None

def video_tile(row: pd.Series, tile_width: int = 640, tile_height: int = 140) -> np.ndarray:
    tile = np.full((tile_height, tile_width, 3), 24, dtype=np.uint8)
    cap = cv2.VideoCapture(str(row.video_path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else 0.0
    timestamps = np.linspace(float(row.max_window_start), float(row.max_window_end), num=FRAMES_PER_VISUALIZATION, endpoint=False)
    thumbnail_width, thumbnail_height = 160, 90
    for index, timestamp in enumerate(timestamps):
        image = read_bgr_at_timestamp(cap, float(timestamp), fps) if cap.isOpened() else None
        if image is None:
            image = np.full((thumbnail_height, thumbnail_width, 3), 80, dtype=np.uint8)
        else:
            image = cv2.resize(image, (thumbnail_width, thumbnail_height), interpolation=cv2.INTER_AREA)
        left = index * thumbnail_width
        tile[32:32 + thumbnail_height, left:left + thumbnail_width] = image
    cap.release()
    caption = f'{row.error_type}  id={row.video_id}  raw={row.raw_video_probability:.2f}  window={row.max_window_start:.1f}-{row.max_window_end:.1f}s'
    cv2.putText(tile, caption, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (235, 235, 235), 1, cv2.LINE_AA)
    return tile

def make_montage(error_name: str, rows: pd.DataFrame, path: Path, columns: int = 3) -> None:
    selected_rows = rows.head(MAX_EXAMPLES_PER_ERROR_TYPE).reset_index(drop=True)
    if selected_rows.empty:
        return
    tiles = [video_tile(row) for _, row in selected_rows.iterrows()]
    blank = np.full_like(tiles[0], 24)
    while len(tiles) % columns:
        tiles.append(blank.copy())
    montage_rows = [cv2.hconcat(tiles[index:index + columns]) for index in range(0, len(tiles), columns)]
    montage = cv2.vconcat(montage_rows)
    cv2.imwrite(str(path), montage, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f'{error_name} montage: {path}')

false_positives = analysis.loc[analysis['error_type'].eq('false_positive')].sort_values('raw_video_probability', ascending=False)
false_negatives = analysis.loc[analysis['error_type'].eq('false_negative')].sort_values('raw_video_probability', ascending=True)
make_montage('False positive', false_positives, ERROR_IMAGE_DIR / 'a2_multipos_false_positive_montage.jpg')
make_montage('False negative', false_negatives, ERROR_IMAGE_DIR / 'a2_multipos_false_negative_montage.jpg')

False positive montage: P:\NexarCollisionData\inference_v2\error_analysis_a2_multipos_final\a2_multipos_false_positive_montage.jpg
False negative montage: P:\NexarCollisionData\inference_v2\error_analysis_a2_multipos_final\a2_multipos_false_negative_montage.jpg


In [5]:
import time
import torch
from torch import nn
from torchvision import models

CHECKPOINT_PATH = MODEL_DIR / 'resnet18_meanmax_pooling_frozen_multipos_best.pt'
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 2.5
NUM_FRAMES = 16
TARGET_HEIGHT, TARGET_WIDTH = 224, 320
FEATURE_DIM = 512
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ResNet18MeanMaxPoolingHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(nn.LayerNorm(FEATURE_DIM * 2), nn.Dropout(0.35), nn.Linear(FEATURE_DIM * 2, 1))

    def forward(self, sequence_features: torch.Tensor) -> torch.Tensor:
        return self.classifier(torch.cat([sequence_features.mean(dim=1), sequence_features.max(dim=1).values], dim=1)).squeeze(1)

weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone = models.resnet18(weights=weights)
encoder = nn.Sequential(*list(backbone.children())[:-1]).to(device).eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
head = ResNet18MeanMaxPoolingHead().to(device).eval()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
head.load_state_dict(checkpoint['model_state_dict'])
print({'device': str(device), 'checkpoint_epoch': checkpoint['epoch'], 'aggregation': selected_aggregation, 'raw_threshold': raw_threshold})

{'device': 'cpu', 'checkpoint_epoch': 12, 'aggregation': 'top3_mean', 'raw_threshold': 0.4}


In [6]:
def resize_letterbox_rgb(image_rgb: np.ndarray) -> np.ndarray:
    height, width = image_rgb.shape[:2]
    scale = min(TARGET_WIDTH / width, TARGET_HEIGHT / height)
    new_width, new_height = max(1, int(round(width * scale))), max(1, int(round(height * scale)))
    resized = cv2.resize(image_rgb, (new_width, new_height), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    pad_x, pad_y = TARGET_WIDTH - new_width, TARGET_HEIGHT - new_height
    return cv2.copyMakeBorder(resized, pad_y // 2, pad_y - pad_y // 2, pad_x // 2, pad_x - pad_x // 2, cv2.BORDER_REPLICATE)

def normalized_tensor(image_rgb: np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(image_rgb.copy()).permute(2, 0, 1).float().div_(255.0)
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def window_starts(duration: float) -> list[float]:
    if duration < WINDOW_SECONDS:
        return []
    starts = list(np.arange(0.0, duration - WINDOW_SECONDS + 1e-8, WINDOW_STRIDE_SECONDS))
    final_start = duration - WINDOW_SECONDS
    if not starts or not np.isclose(starts[-1], final_start):
        starts.append(final_start)
    return sorted({round(float(start), 6) for start in starts})

def decode_rgb_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float):
    step = 1.0 / fps if np.isfinite(fps) and fps > 0 else 1.0 / 30.0
    for attempt, offset in enumerate((0.0, step, -step, 2 * step)):
        target_timestamp = max(0.0, timestamp + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, target_timestamp * 1000.0)
        ok, image_bgr = cap.read()
        if ok and image_bgr is not None:
            return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), ('exact' if attempt == 0 else f'seek_fallback_{attempt}')
    return None, 'decode_failed'

def aggregate_probabilities(probabilities: np.ndarray) -> float:
    ordered = np.sort(probabilities)[::-1]
    if selected_aggregation == 'max':
        return float(ordered[0])
    if selected_aggregation == 'top2_mean':
        return float(ordered[:min(2, len(ordered))].mean())
    if selected_aggregation == 'top3_mean':
        return float(ordered[:min(3, len(ordered))].mean())
    if selected_aggregation == 'mean':
        return float(ordered.mean())
    raise ValueError(f'Unknown aggregation: {selected_aggregation}')

def calibrate_probability(probability: float) -> float:
    probability = float(np.clip(probability, 1e-6, 1.0 - 1e-6))
    logit = math.log(probability / (1.0 - probability))
    return float(1.0 / (1.0 + math.exp(-(logit / temperature))))

@torch.inference_mode()
def predict_full_mp4_final(video_path: str | Path) -> dict:
    """Classify an arbitrary MP4 without label, event time or metadata."""
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open MP4: {video_path}')
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps if fps > 0 else np.nan
    if not np.isfinite(duration) or duration < WINDOW_SECONDS:
        cap.release()
        raise ValueError(f'Video must be at least {WINDOW_SECONDS} seconds long.')

    records = []
    started = time.perf_counter()
    try:
        for start in window_starts(duration):
            frames, statuses, previous_frame = [], [], None
            for timestamp in np.linspace(start, start + WINDOW_SECONDS, num=NUM_FRAMES, endpoint=False):
                frame_rgb, status = decode_rgb_at_timestamp(cap, float(timestamp), fps)
                if frame_rgb is None and previous_frame is not None:
                    frame_rgb, status = previous_frame.copy(), 'repeated_previous_after_decode_failure'
                if frame_rgb is not None:
                    previous_frame = frame_rgb
                    frames.append(normalized_tensor(resize_letterbox_rgb(frame_rgb)))
                statuses.append(status)
            if len(frames) != NUM_FRAMES:
                raise RuntimeError(f'Could not decode all 16 frames for window {start:.3f}s.')
            features = encoder(torch.stack(frames).to(device)).flatten(1).unsqueeze(0)
            probability = float(torch.sigmoid(head(features)).item())
            records.append({
                'window_start': float(start), 'window_end': float(start + WINDOW_SECONDS),
                'positive_probability': probability, 'decode_status': ';'.join(sorted(set(statuses))),
            })
    finally:
        cap.release()

    window_probabilities = np.asarray([record['positive_probability'] for record in records], dtype=float)
    raw_video_probability = aggregate_probabilities(window_probabilities)
    calibrated_video_probability = calibrate_probability(raw_video_probability)
    best_window = records[int(np.argmax(window_probabilities))]
    return {
        'video_path': str(video_path), 'duration_seconds': float(duration), 'num_windows': len(records),
        'raw_video_probability': raw_video_probability,
        'calibrated_video_probability': calibrated_video_probability,
        'prediction': int(raw_video_probability >= raw_threshold),
        'decision_threshold_raw': raw_threshold,
        'decision_threshold_calibrated_equivalent': calibrated_threshold,
        'uncertain': bool(0.4 <= calibrated_video_probability <= 0.6),
        'aggregation': selected_aggregation,
        'highest_probability_window': [best_window['window_start'], best_window['window_end']],
        'inference_seconds': time.perf_counter() - started,
        'window_predictions': records,
    }

print('predict_full_mp4_final(...) is ready for an arbitrary MP4.')

predict_full_mp4_final(...) is ready for an arbitrary MP4.


In [7]:
RUN_ROUNDTRIP_CHECK = True
EXTERNAL_MP4_PATH: str | None = None

if RUN_ROUNDTRIP_CHECK:
    reference_row = analysis.sort_values('video_id', key=lambda values: values.astype(int)).iloc[0]
    roundtrip = predict_full_mp4_final(reference_row.video_path)
    assert roundtrip['prediction'] == int(reference_row.raw_prediction)
    assert abs(roundtrip['raw_video_probability'] - float(reference_row.raw_video_probability)) < 1e-5
    print('Round-trip check passed on one already-labelled validation MP4. This validates the inference implementation; it is not an independent performance estimate.')
    print({key: value for key, value in roundtrip.items() if key != 'window_predictions'})

if EXTERNAL_MP4_PATH is None:
    print('To classify a genuinely new MP4, set EXTERNAL_MP4_PATH to its full path and run this cell again.')
else:
    external_result = predict_full_mp4_final(EXTERNAL_MP4_PATH)
    print('External MP4 result:')
    print({key: value for key, value in external_result.items() if key != 'window_predictions'})

Round-trip check passed on one already-labelled validation MP4. This validates the inference implementation; it is not an independent performance estimate.
{'video_path': 'P:\\NexarCollisionData\\train\\positive\\00014.mp4', 'duration_seconds': 40.333333333333336, 'num_windows': 16, 'raw_video_probability': 0.5043248037497202, 'calibrated_video_probability': 0.5030706932990563, 'prediction': 1, 'decision_threshold_raw': 0.4, 'decision_threshold_calibrated_equivalent': 0.4285218762661993, 'uncertain': True, 'aggregation': 'top3_mean', 'highest_probability_window': [32.5, 37.5], 'inference_seconds': 86.65751849999651}
To classify a genuinely new MP4, set EXTERNAL_MP4_PATH to its full path and run this cell again.
